# XGBoost image-feature model

Binary classification using fixed image representations.

The notebook supports two interchangeable image-feature sources:

- **PCA** features generated separately for H1 and H2 from the corresponding training split.
- **CNN embeddings** generated with a frozen pretrained TorchVision feature extractor and reused across hypotheses.

Current hypotheses:
- H1: biopsy recommendation
- H2: malignancy among biopsied lesions

Model selection is performed exclusively on the training set using patient-grouped cross-validation.

If training becomes computationally expensive, the training split can be reduced to a reproducible, target-stratified subset of lesions. Validation and test sets are always kept complete.

## 1. Imports and configuration

In [ ]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedGroupKFold,
)
from sklearn.metrics import (
    auc,
    make_scorer,
    precision_recall_curve,
)

from xgboost import XGBClassifier

from skin_lesion_ai.inference.evaluation import (
    evaluate,
    save_model,
)
from skin_lesion_ai.utils.data_utils import (
    load_image_embeddings,
    load_image_pca,
    load_metadata_parquet,
    subsample_training_split,
)


# ---------------------------------------------------------------------
# Main configuration
# ---------------------------------------------------------------------

HYPOTHESIS = 1

# Choose: "pca" or "embedding"
IMAGE_REPRESENTATION = "pca"

IMAGE_SIZE = 136

# PCA configuration
PCA_N_COMPONENTS = 128
PCA_TIMESTAMP = None

# Embedding configuration
# Choose: "efficientnet_b0", "resnet50", "densenet121", "convnext_tiny",
EMBEDDING_MODEL = "efficientnet_b0"
EMBEDDING_TIMESTAMP = None

# Reproducible training-size control.
# None -> use the complete training split.
# Example: 3000 -> X patients are selected so the sum of their lesions is <= 3000 and a similar class distribution is maintained.
# The training split is then subsampled to only include those patients.
TRAIN_MAX_LESIONS = None

RANDOM_STATE = 42
TRAIN_SUBSAMPLE_RANDOM_STATE = 42

N_CV_SPLITS = 5
N_RANDOM_ITER = 50


# ---------------------------------------------------------------------
# Hypothesis configuration
# ---------------------------------------------------------------------

HYPOTHESIS_CONFIG = {
    1: {
        "target": "target_biopsy",
        "split_suffix": "h1",
        "description": "Biopsy recommendation",
    },
    2: {
        "target": "target_malignant",
        "split_suffix": "h2",
        "description": "Malignancy among biopsied lesions",
    },
}

SUPPORTED_REPRESENTATIONS = {"pca", "embedding"}
SUPPORTED_EMBEDDING_MODELS = {
    "efficientnet_b0",
    "resnet50",
    "densenet121",
    "convnext_tiny",
}

if HYPOTHESIS not in HYPOTHESIS_CONFIG:
    raise ValueError("HYPOTHESIS must be either 1 or 2.")

if IMAGE_REPRESENTATION not in SUPPORTED_REPRESENTATIONS:
    raise ValueError(
        f"IMAGE_REPRESENTATION must be one of: {sorted(SUPPORTED_REPRESENTATIONS)}"
    )

if (
    IMAGE_REPRESENTATION == "embedding"
    and EMBEDDING_MODEL not in SUPPORTED_EMBEDDING_MODELS
):
    raise ValueError(
        f"Unsupported EMBEDDING_MODEL. Choose one of: "
        f"{sorted(SUPPORTED_EMBEDDING_MODELS)}"
    )

TARGET = HYPOTHESIS_CONFIG[HYPOTHESIS]["target"]
SPLIT_SUFFIX = HYPOTHESIS_CONFIG[HYPOTHESIS]["split_suffix"]
HYPOTHESIS_DESCRIPTION = HYPOTHESIS_CONFIG[HYPOTHESIS]["description"]

if IMAGE_REPRESENTATION == "pca":
    REPRESENTATION_NAME = f"pca_{PCA_N_COMPONENTS}c"
else:
    REPRESENTATION_NAME = f"embedding_{EMBEDDING_MODEL}"

TRAIN_SIZE_TAG = "full_train" if TRAIN_MAX_LESIONS is None else f"n{TRAIN_MAX_LESIONS}"

MODEL_NAME = f"xgboost_image_{REPRESENTATION_NAME}_h{HYPOTHESIS}_{TRAIN_SIZE_TAG}"

print(f"Hypothesis {HYPOTHESIS}: {HYPOTHESIS_DESCRIPTION}")
print(f"Target: {TARGET}")
print(f"Image representation: {REPRESENTATION_NAME}")
print(f"Training size setting: {TRAIN_SIZE_TAG}")
print(f"Model name: {MODEL_NAME}")

Hypothesis 1: Biopsy recommendation
Target: target_biopsy
Image representation: pca_128c
Training size setting: full_train
Model name: xgboost_image_pca_128c_h1_full_train


## 2. Load train, validation and test metadata

The original H1/H2 train, validation and test splits are loaded first.

If `TRAIN_MAX_LESIONS` is set, **only the training split** is reduced. The helper in `data_utils.py` performs target-stratified lesion-level sampling after sorting by `isic_id`, making the selected subset reproducible across notebooks and image representations for the same hypothesis, sample size and seed.

Validation and test remain unchanged.

In [2]:
df_train_full = load_metadata_parquet(
    stage="processed",
    filename=f"train_split_{SPLIT_SUFFIX}",
    timestamp_flag=True,
)

df_validation = load_metadata_parquet(
    stage="processed",
    filename=f"val_split_{SPLIT_SUFFIX}",
    timestamp_flag=True,
)

df_test = load_metadata_parquet(
    stage="processed",
    filename=f"test_split_{SPLIT_SUFFIX}",
    timestamp_flag=True,
)


df_train = subsample_training_split(
    df=df_train_full,
    n_samples=TRAIN_MAX_LESIONS,
    target_column=TARGET,
    id_column="isic_id",
    random_state=TRAIN_SUBSAMPLE_RANDOM_STATE,
)


split_summary = pd.DataFrame(
    {
        "split": [
            "train_full",
            "train_used",
            "validation",
            "test",
        ],
        "lesions": [
            len(df_train_full),
            len(df_train),
            len(df_validation),
            len(df_test),
        ],
        "patients": [
            df_train_full["patient_id"].nunique(),
            df_train["patient_id"].nunique(),
            df_validation["patient_id"].nunique(),
            df_test["patient_id"].nunique(),
        ],
        "positive_rate": [
            df_train_full[TARGET].mean(),
            df_train[TARGET].mean(),
            df_validation[TARGET].mean(),
            df_test[TARGET].mean(),
        ],
    }
)

display(split_summary)

,split,lesions,patients,positive_rate
0,train_full,305024,784,0.002662
1,train_used,305024,784,0.002662
2,validation,38131,94,0.002649
3,test,38125,99,0.002623


## 3. Load image representation

The same modelling code is used for PCA and CNN embeddings.

For embeddings, `load_image_embeddings()` reads the selected extractor run and retains only the lesion identifiers required by each split.

For PCA, `load_image_pca()` reads the H1- or H2-specific PCA run. When the training split has been subsampled, only the selected training lesion identifiers are retained.

Both representations use columns named `feature_XXXX`, so downstream XGBoost code is representation-agnostic.

In [3]:
def load_features_for_split(
    split_name: str,
    metadata_df: pd.DataFrame,
) -> pd.DataFrame:
    """Load the selected image representation for one metadata split."""

    lesion_ids = metadata_df["isic_id"].astype(str)

    if IMAGE_REPRESENTATION == "embedding":
        return load_image_embeddings(
            model_name=EMBEDDING_MODEL,
            image_size=IMAGE_SIZE,
            timestamp_value=EMBEDDING_TIMESTAMP,
            lesion_ids=lesion_ids,
        )

    return load_image_pca(
        hypothesis=HYPOTHESIS,
        split=split_name,
        image_size=IMAGE_SIZE,
        n_components=PCA_N_COMPONENTS,
        timestamp_value=PCA_TIMESTAMP,
        lesion_ids=lesion_ids,
    )


train_image_features = load_features_for_split(
    split_name="train",
    metadata_df=df_train,
)

validation_image_features = load_features_for_split(
    split_name="val",
    metadata_df=df_validation,
)

test_image_features = load_features_for_split(
    split_name="test",
    metadata_df=df_test,
)


print(f"Train image features: {train_image_features.shape}")
print(f"Validation image features: {validation_image_features.shape}")
print(f"Test image features: {test_image_features.shape}")

Train image features: (305024, 130)
Validation image features: (38131, 130)
Test image features: (38125, 130)


## 4. Validate and align metadata with image features

In [4]:
ID_COLUMN = "isic_id"
IMAGE_ID_COLUMN = "lesion_id"
GROUP_COLUMN = "patient_id"


def merge_metadata_and_image_features(
    metadata_df: pd.DataFrame,
    image_df: pd.DataFrame,
    split_name: str,
) -> tuple[pd.DataFrame, list[str]]:
    """Align one metadata split with its image-feature rows by lesion ID."""

    required_metadata_columns = {
        ID_COLUMN,
        GROUP_COLUMN,
        TARGET,
    }

    missing_metadata = required_metadata_columns.difference(metadata_df.columns)

    if missing_metadata:
        raise KeyError(
            f"{split_name} metadata are missing columns: {sorted(missing_metadata)}"
        )

    if IMAGE_ID_COLUMN not in image_df.columns:
        raise KeyError(f"{split_name} image features are missing '{IMAGE_ID_COLUMN}'.")

    feature_columns = sorted(
        column for column in image_df.columns if column.startswith("feature_")
    )

    if not feature_columns:
        raise ValueError(f"No feature_XXXX columns found in {split_name} image data.")

    metadata = metadata_df.copy()
    features = image_df.copy()

    metadata[ID_COLUMN] = metadata[ID_COLUMN].astype(str)
    metadata[GROUP_COLUMN] = metadata[GROUP_COLUMN].astype(str)
    features[IMAGE_ID_COLUMN] = features[IMAGE_ID_COLUMN].astype(str)

    if not metadata[ID_COLUMN].is_unique:
        raise ValueError(
            f"{split_name} metadata contain duplicated {ID_COLUMN} values."
        )

    if not features[IMAGE_ID_COLUMN].is_unique:
        raise ValueError(
            f"{split_name} image data contain duplicated {IMAGE_ID_COLUMN} values."
        )

    metadata_ids = set(metadata[ID_COLUMN])
    feature_ids = set(features[IMAGE_ID_COLUMN])

    if metadata_ids != feature_ids:
        missing_features = metadata_ids.difference(feature_ids)
        unexpected_features = feature_ids.difference(metadata_ids)

        raise ValueError(
            f"{split_name} metadata/image ID mismatch. "
            f"Missing features: {len(missing_features):,}; "
            f"unexpected features: {len(unexpected_features):,}."
        )

    # Validate patient identifiers when the image representation stores them.
    if GROUP_COLUMN in features.columns:
        patient_check = metadata[[ID_COLUMN, GROUP_COLUMN]].merge(
            features[[IMAGE_ID_COLUMN, GROUP_COLUMN]],
            left_on=ID_COLUMN,
            right_on=IMAGE_ID_COLUMN,
            how="inner",
            validate="one_to_one",
            suffixes=("_metadata", "_image"),
        )

        mismatch_mask = patient_check[f"{GROUP_COLUMN}_metadata"].astype(
            str
        ) != patient_check[f"{GROUP_COLUMN}_image"].astype(str)

        if mismatch_mask.any():
            raise ValueError(f"Patient-ID mismatch detected in {split_name}.")

    feature_table = features[[IMAGE_ID_COLUMN, *feature_columns]].copy()

    merged = metadata.merge(
        feature_table,
        left_on=ID_COLUMN,
        right_on=IMAGE_ID_COLUMN,
        how="left",
        validate="one_to_one",
    ).drop(columns=IMAGE_ID_COLUMN)

    if merged[feature_columns].isna().any().any():
        raise ValueError(f"Missing image-feature values found in {split_name}.")

    return merged, feature_columns


df_train_model, FEATURES = merge_metadata_and_image_features(
    metadata_df=df_train,
    image_df=train_image_features,
    split_name="train",
)

df_validation_model, validation_features = merge_metadata_and_image_features(
    metadata_df=df_validation,
    image_df=validation_image_features,
    split_name="validation",
)

df_test_model, test_features = merge_metadata_and_image_features(
    metadata_df=df_test,
    image_df=test_image_features,
    split_name="test",
)

if FEATURES != validation_features or FEATURES != test_features:
    raise ValueError("Image feature columns differ across train, validation and test.")

print(f"Number of image features: {len(FEATURES):,}")
print(f"First feature: {FEATURES[0]}")
print(f"Last feature: {FEATURES[-1]}")

Number of image features: 128
First feature: feature_0000
Last feature: feature_0127


## 5. Prepare modelling matrices

The image representation is used directly as the XGBoost predictor matrix.

Patient identifiers are retained as grouping labels for `StratifiedGroupKFold`, ensuring that lesions from the same patient cannot appear in both the training and validation portions of one cross-validation fold.

In [5]:
# Training data
X_train = df_train_model[FEATURES].copy()
y_train = df_train_model[TARGET].astype(int).copy()
groups_train = df_train_model[GROUP_COLUMN].copy()

# Validation data
X_validation = df_validation_model[FEATURES].copy()
y_validation = df_validation_model[TARGET].astype(int).copy()

# Test data
X_test = df_test_model[FEATURES].copy()
y_test = df_test_model[TARGET].astype(int).copy()


print("Training:")
print(f"  X: {X_train.shape}")
print(f"  y: {y_train.shape}")
print(f"  patients: {groups_train.nunique():,}")

print("\nValidation:")
print(f"  X: {X_validation.shape}")
print(f"  y: {y_validation.shape}")

print("\nTest:")
print(f"  X: {X_test.shape}")
print(f"  y: {y_test.shape}")

Training:
  X: (305024, 128)
  y: (305024,)
  patients: 784

Validation:
  X: (38131, 128)
  y: (38131,)

Test:
  X: (38125, 128)
  y: (38125,)


## 6. Cross-validation and scoring strategy

Hyperparameter optimization is performed exclusively within the selected training data.

The same patient-grouped cross-validation strategy and scoring hierarchy as the metadata XGBoost model are used:

1. **PR-AUC** as the primary selection criterion.
2. **ROC-AUC** as the first tie-breaker.
3. **Brier score** as the second tie-breaker.

PR-AUC is computed as the trapezoidal area under the Precision-Recall curve, matching the final evaluation pipeline.

No classification threshold is selected during cross-validation.

In [6]:
# ---------------------------------------------------------------------
# Cross-validation
# ---------------------------------------------------------------------

cv = StratifiedGroupKFold(
    n_splits=N_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_splits = list(
    cv.split(
        X=X_train,
        y=y_train,
        groups=groups_train,
    )
)


# ---------------------------------------------------------------------
# Validate CV folds
# ---------------------------------------------------------------------

cv_summary = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1,
):
    train_patients = set(groups_train.iloc[train_idx])
    val_patients = set(groups_train.iloc[val_idx])

    if train_patients.intersection(val_patients):
        raise ValueError(f"Patient leakage detected in CV fold {fold}.")

    train_classes = set(y_train.iloc[train_idx].unique())
    val_classes = set(y_train.iloc[val_idx].unique())

    if train_classes != {0, 1} or val_classes != {0, 1}:
        raise ValueError(
            f"CV fold {fold} does not contain both target classes. "
            "Increase TRAIN_MAX_LESIONS or reduce N_CV_SPLITS."
        )

    cv_summary.append(
        {
            "fold": fold,
            "train_lesions": len(train_idx),
            "validation_lesions": len(val_idx),
            "train_patients": len(train_patients),
            "validation_patients": len(val_patients),
            "train_positive_rate": y_train.iloc[train_idx].mean(),
            "validation_positive_rate": y_train.iloc[val_idx].mean(),
        }
    )

cv_summary = pd.DataFrame(cv_summary)

display(cv_summary)

,fold,train_lesions,validation_lesions,train_patients,validation_patients,train_positive_rate,validation_positive_rate
0,1,244020,61004,628,156,0.002668,0.002639
1,2,244021,61003,628,156,0.002668,0.002639
2,3,244014,61010,627,157,0.002639,0.002754
3,4,244020,61004,626,158,0.002668,0.002639
4,5,244021,61003,627,157,0.002668,0.002639


In [7]:
# ---------------------------------------------------------------------
# Scoring functions
# ---------------------------------------------------------------------


def trapezoidal_pr_auc(y_true, y_prob):
    """Compute trapezoidal area under the Precision-Recall curve."""

    precision, recall, _ = precision_recall_curve(
        y_true,
        y_prob,
    )

    return auc(recall, precision)


pr_auc_scorer = make_scorer(
    trapezoidal_pr_auc,
    response_method="predict_proba",
)


SCORING = {
    "pr_auc": pr_auc_scorer,
    "roc_auc": "roc_auc",
    "brier": "neg_brier_score",
}


def select_best_candidate(cv_results):
    """Select the best candidate using PR-AUC > ROC-AUC > Brier score."""

    results = pd.DataFrame(cv_results)

    ranked_results = results.sort_values(
        by=[
            "mean_test_pr_auc",
            "mean_test_roc_auc",
            "mean_test_brier",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        na_position="last",
    )

    return int(ranked_results.index[0])

## 7. Broad hyperparameter search

A randomized search explores the principal XGBoost dimensions controlling boosting, tree complexity, sampling, regularization and class imbalance.

If runtime is excessive, reduce `TRAIN_MAX_LESIONS` in the configuration cell rather than manually sampling inside this section. This keeps the same reproducible training subset available to PCA and embedding experiments.

In [8]:
# ---------------------------------------------------------------------
# Class imbalance
# ---------------------------------------------------------------------

n_negative = int((y_train == 0).sum())
n_positive = int((y_train == 1).sum())

class_ratio = n_negative / n_positive

scale_pos_weight_values = sorted(
    {
        1.0,
        round(np.sqrt(class_ratio), 4),
        round(class_ratio, 4),
    }
)

print(f"Negative lesions: {n_negative:,}")
print(f"Positive lesions: {n_positive:,}")
print(f"Negative / positive ratio: {class_ratio:.3f}")
print(f"scale_pos_weight candidates: {scale_pos_weight_values}")

Negative lesions: 304,212
Positive lesions: 812
Negative / positive ratio: 374.645
scale_pos_weight candidates: [1.0, 19.3558, 374.6453]


`scale_pos_weight` controls the relative contribution of positive-class observations to the XGBoost loss function.

Three values are explored: no weighting (`1`), an intermediate correction (`sqrt(n_negative / n_positive)`), and the full negative-to-positive class ratio.

In [9]:
# ---------------------------------------------------------------------
# Base XGBoost classifier
# ---------------------------------------------------------------------

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
)


# ---------------------------------------------------------------------
# Broad search space
# ---------------------------------------------------------------------

broad_param_space = {
    "n_estimators": [
        200,
        400,
        600,
        800,
        1000,
        1400,
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05,
        0.10,
        0.20,
    ],
    "max_depth": [
        2,
        3,
        4,
        5,
        6,
    ],
    "min_child_weight": [
        1,
        3,
        5,
        10,
        20,
    ],
    "subsample": [
        0.6,
        0.8,
        1.0,
    ],
    "colsample_bytree": [
        0.6,
        0.8,
        1.0,
    ],
    "gamma": [
        0.0,
        0.1,
        0.5,
        1.0,
        2.0,
        5.0,
    ],
    "reg_alpha": [
        0.0,
        0.01,
        0.1,
        1.0,
        5.0,
    ],
    "reg_lambda": [
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
    ],
    "scale_pos_weight": scale_pos_weight_values,
}

In [10]:
# ---------------------------------------------------------------------
# Broad randomized search
# ---------------------------------------------------------------------

broad_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=broad_param_space,
    n_iter=N_RANDOM_ITER,
    scoring=SCORING,
    refit=select_best_candidate,
    cv=cv_splits,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    pre_dispatch="n_jobs",
    verbose=2,
    return_train_score=False,
    error_score="raise",
)

broad_search.fit(
    X_train,
    y_train,
)

Fitting 5 folds for each of 150 candidates, totalling 750 fits


KeyboardInterrupt: 

## 8. Refined hyperparameter search

The broad-search optimum defines a smaller local grid over boosting capacity and class weighting. Sampling and regularization parameters are fixed at their best broad-search values.

In [ ]:
# ---------------------------------------------------------------------
# Best broad-search configuration
# ---------------------------------------------------------------------

broad_best_params = broad_search.best_params_

broad_results = pd.DataFrame(broad_search.cv_results_)
broad_best_row = broad_results.loc[broad_search.best_index_]

print("Broad search best CV performance:")
print(
    f"PR-AUC: "
    f"{broad_best_row['mean_test_pr_auc']:.5f} "
    f"± {broad_best_row['std_test_pr_auc']:.5f}"
)
print(
    f"ROC-AUC: "
    f"{broad_best_row['mean_test_roc_auc']:.5f} "
    f"± {broad_best_row['std_test_roc_auc']:.5f}"
)
print(f"Brier score: {-broad_best_row['mean_test_brier']:.5f}")

print("\nBest broad-search parameters:")
display(broad_best_params)

In [ ]:
# ---------------------------------------------------------------------
# Build refined grid around broad-search optimum
# ---------------------------------------------------------------------

best_n_estimators = broad_best_params["n_estimators"]
best_learning_rate = broad_best_params["learning_rate"]
best_max_depth = broad_best_params["max_depth"]
best_min_child_weight = broad_best_params["min_child_weight"]
best_scale_pos_weight = broad_best_params["scale_pos_weight"]


refined_param_grid = {
    "n_estimators": sorted(
        {
            max(
                100,
                int(round(best_n_estimators * 0.75)),
            ),
            best_n_estimators,
            int(round(best_n_estimators * 1.25)),
        }
    ),
    "learning_rate": sorted(
        {
            max(
                0.005,
                best_learning_rate * 0.7,
            ),
            best_learning_rate,
            min(
                0.30,
                best_learning_rate * 1.3,
            ),
        }
    ),
    "max_depth": sorted(
        {
            max(1, best_max_depth - 1),
            best_max_depth,
            best_max_depth + 1,
        }
    ),
    "min_child_weight": sorted(
        {
            max(
                1,
                best_min_child_weight / 2,
            ),
            best_min_child_weight,
            best_min_child_weight * 2,
        }
    ),
    "scale_pos_weight": sorted(
        {
            max(
                0.1,
                best_scale_pos_weight * 0.75,
            ),
            best_scale_pos_weight,
            best_scale_pos_weight * 1.25,
        }
    ),
}

refined_param_grid

In [ ]:
# ---------------------------------------------------------------------
# Fix broad-search sampling and regularization parameters
# ---------------------------------------------------------------------

refined_xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
    subsample=broad_best_params["subsample"],
    colsample_bytree=broad_best_params["colsample_bytree"],
    gamma=broad_best_params["gamma"],
    reg_alpha=broad_best_params["reg_alpha"],
    reg_lambda=broad_best_params["reg_lambda"],
)

In [ ]:
# ---------------------------------------------------------------------
# Refined exhaustive search
# ---------------------------------------------------------------------

refined_search = GridSearchCV(
    estimator=refined_xgb_model,
    param_grid=refined_param_grid,
    scoring=SCORING,
    refit=select_best_candidate,
    cv=cv_splits,
    n_jobs=-1,
    pre_dispatch="n_jobs",
    verbose=2,
    return_train_score=False,
    error_score="raise",
)

refined_search.fit(
    X_train,
    y_train,
)

## 9. Cross-validation results and final model

The broad and refined searches are compared using the same predefined hierarchy:

**PR-AUC → ROC-AUC → Brier score.**

The estimator selected by the refined `GridSearchCV` is automatically refitted on the complete **selected training set**.

In [ ]:
# ---------------------------------------------------------------------
# Collect CV results
# ---------------------------------------------------------------------

broad_cv_results = pd.DataFrame(broad_search.cv_results_)

refined_cv_results = pd.DataFrame(refined_search.cv_results_)


def get_best_search_result(
    search,
    search_name,
):
    """Return the selected CV result from one search stage."""

    results = pd.DataFrame(search.cv_results_)
    row = results.loc[search.best_index_]

    return {
        "search": search_name,
        "pr_auc": row["mean_test_pr_auc"],
        "pr_auc_std": row["std_test_pr_auc"],
        "roc_auc": row["mean_test_roc_auc"],
        "roc_auc_std": row["std_test_roc_auc"],
        "brier_score": -row["mean_test_brier"],
        "brier_score_std": row["std_test_brier"],
    }


search_comparison = pd.DataFrame(
    [
        get_best_search_result(
            broad_search,
            "Broad randomized search",
        ),
        get_best_search_result(
            refined_search,
            "Refined grid search",
        ),
    ]
)

display(search_comparison)

In [ ]:
# ---------------------------------------------------------------------
# Top refined-search candidates
# ---------------------------------------------------------------------

top_refined_models = refined_cv_results.copy()

top_refined_models["mean_brier_score"] = -top_refined_models["mean_test_brier"]

top_refined_models = (
    top_refined_models.sort_values(
        by=[
            "mean_test_pr_auc",
            "mean_test_roc_auc",
            "mean_test_brier",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )[
        [
            "params",
            "mean_test_pr_auc",
            "std_test_pr_auc",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_brier_score",
            "std_test_brier",
            "mean_fit_time",
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

display(top_refined_models)

In [ ]:
# ---------------------------------------------------------------------
# Final model
# ---------------------------------------------------------------------

best_model = refined_search.best_estimator_

final_model_params = {
    **broad_best_params,
    **refined_search.best_params_,
}

print("Final selected hyperparameters:")
display(final_model_params)

print(f"Final model automatically refitted on {len(X_train):,} training lesions.")

## 10. Save final model and experiment configuration

Besides the trained XGBoost model, the run directory stores the exact training lesion identifiers and a small experiment-configuration JSON. This makes reduced-training experiments auditable and allows the same subset to be checked across image representations.

In [ ]:
model_directory = save_model(
    model=best_model,
    model_name=MODEL_NAME,
    model_type=1,
)


# Save exact training IDs used by this run.
training_ids_path = Path(model_directory) / "training_lesion_ids.csv"

df_train[[ID_COLUMN]].to_csv(
    training_ids_path,
    index=False,
)


def to_python_scalar(value):
    """Convert NumPy scalar values before JSON serialization."""

    if isinstance(value, np.generic):
        return value.item()

    return value


experiment_configuration = {
    "hypothesis": HYPOTHESIS,
    "target": TARGET,
    "image_representation": IMAGE_REPRESENTATION,
    "image_size": IMAGE_SIZE,
    "pca_n_components": (PCA_N_COMPONENTS if IMAGE_REPRESENTATION == "pca" else None),
    "pca_timestamp": (PCA_TIMESTAMP if IMAGE_REPRESENTATION == "pca" else None),
    "embedding_model": (
        EMBEDDING_MODEL if IMAGE_REPRESENTATION == "embedding" else None
    ),
    "embedding_timestamp": (
        EMBEDDING_TIMESTAMP if IMAGE_REPRESENTATION == "embedding" else None
    ),
    "train_full_lesions": len(df_train_full),
    "train_used_lesions": len(df_train),
    "train_max_lesions_setting": TRAIN_MAX_LESIONS,
    "train_subsample_random_state": TRAIN_SUBSAMPLE_RANDOM_STATE,
    "cv_random_state": RANDOM_STATE,
    "n_cv_splits": N_CV_SPLITS,
    "n_random_iter": N_RANDOM_ITER,
    "n_features": len(FEATURES),
    "final_model_params": {
        key: to_python_scalar(value) for key, value in final_model_params.items()
    },
}

experiment_config_path = Path(model_directory) / "training_configuration.json"

with experiment_config_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_configuration,
        file,
        indent=2,
    )


print(f"Model directory: {model_directory}")
print(f"Training IDs: {training_ids_path}")
print(f"Training configuration: {experiment_config_path}")

## 11. Generate prediction probabilities

In [ ]:
train_probabilities = best_model.predict_proba(X_train)[:, 1]

validation_probabilities = best_model.predict_proba(X_validation)[:, 1]


df_train_predictions = pd.DataFrame(
    {
        "isic_id": df_train[ID_COLUMN].to_numpy(),
        "probability": train_probabilities,
    }
)

df_validation_predictions = pd.DataFrame(
    {
        "isic_id": df_validation[ID_COLUMN].to_numpy(),
        "probability": validation_probabilities,
    }
)

display(df_train_predictions.head())
display(df_validation_predictions.head())

## 12. Train and validation evaluation

Threshold-independent discrimination and calibration metrics are calculated for the selected training set and the complete validation set.

The clinical classification threshold is selected exclusively from validation by maximizing specificity while maintaining the predefined minimum sensitivity. The selected threshold is then applied unchanged to both training and validation.

In [ ]:
results = evaluate(
    df_train_predictions=df_train_predictions,
    df_validation_predictions=df_validation_predictions,
    df_train=df_train,
    df_validation=df_validation,
    hypothesis=HYPOTHESIS,
    model_directory=model_directory,
    target_sensitivity=0.95,
)

display(results["summary"])

In [ ]:
from IPython.display import (
    Image,
    Markdown,
    display,
)

for figure_name, figure_path in results["figure_paths"].items():
    display(Markdown(f"### {figure_name.replace('_', ' ').title()}"))
    display(Image(filename=str(figure_path)))

## Usage summary

To rerun comparable experiments, change only the configuration cell:

- `HYPOTHESIS = 1` or `2`.
- `IMAGE_REPRESENTATION = "pca"` or `"embedding"`.
- For embeddings, select `EMBEDDING_MODEL`.
- For PCA, select `PCA_N_COMPONENTS`.
- Set `TRAIN_MAX_LESIONS = None` for the full training split, or an integer such as `3000` for a reproducible reduced training set.

With the same hypothesis, `TRAIN_MAX_LESIONS` and `TRAIN_SUBSAMPLE_RANDOM_STATE`, the same training lesion identifiers are selected across PCA and embedding experiments.

PCA itself remains fitted on the complete original training split used during PCA generation; reducing `TRAIN_MAX_LESIONS` only reduces the downstream XGBoost training set. This is appropriate when subsampling is used to control XGBoost runtime rather than to redefine the PCA fitting population.